# Réseaux de Neurones sur Graphes (GNN)

> Version française du notebook — basée sur la session d'apprentissage.

## 1. Définition

**Les Réseaux de Neurones sur Graphes (GNN)** sont une famille de modèles de deep learning conçus pour opérer directement sur des données structurées en graphe. Plutôt que d'imposer une disposition euclidienne fixe (comme pour les images ou les séquences), les GNN apprennent des représentations en propagant et agrégeant de l'information le long des arêtes du graphe.

### Historique

| Époque | Contribution |
|--------|--------------|
| 2005–2009 | Gori et al. et Scarselli et al. — première formulation GNN par itérations à point fixe |
| 2017 | Kipf & Welling — Graph Convolutional Networks (GCN), approche spectrale |
| 2017 | Hamilton et al. — GraphSAGE, embeddings de nœuds inductifs par échantillonnage de voisinage |
| 2018 | Veličković et al. — Graph Attention Networks (GAT), agrégation pondérée par attention |
| 2019 | Joshi et al. — GCN appliqué à la prédiction d'arêtes TSP |

### Catégorie

Les GNN appartiennent au **geometric deep learning** — une généralisation des CNN aux domaines non euclidiens. Pour l'optimisation combinatoire, ils sont utilisés comme **heuristiques de construction** (scoring d'arêtes → décodage de tournée) ou **heuristiques d'amélioration** (recherche locale apprise).

### Concepts clés

- **Embedding de nœud** $h_v^{(l)}$ : représentation vectorielle du nœud $v$ à la couche $l$, encodant la structure locale du voisinage.
- **Message passing** : les nœuds agrègent itérativement les features de leurs voisins.
- **Readout / pooling** : prédiction au niveau du graphe ou des arêtes à partir des embeddings de nœuds.
- **Invariance par permutation** : la sortie est invariante à l'ordre des nœuds, ce qui est naturel pour les problèmes combinatoires.

### Pourquoi le deep learning pour le routage ?

Un solveur classique ou une méta-heuristique doit **ré-exécuter** tout l'algorithme à chaque nouvelle instance. Un modèle GNN, lui, sépare deux phases :

| Phase | Quand | Coût |
|-------|-------|------|
| **Entraînement** (*training*) | Une seule fois, hors ligne | Élevé (heures/jours GPU) |
| **Inférence** (*inference*) | À chaque nouvelle instance | Faible (millisecondes) |

La capacité à bien performer sur des instances jamais vues s'appelle la **généralisation**. C'est le défi central : un modèle entraîné sur $n=50$ peut échouer sur $n=200$.

## 2. Description Formelle

### Représentation du graphe

Un graphe pondéré est défini comme $\mathcal{G} = (\mathcal{V}, \mathcal{E}, \mathbf{X}, \mathbf{E})$ où :

| Symbole | Type | Signification |
|---------|------|---------------|
| $\mathcal{V} = \{v_1, \dots, v_n\}$ | Ensemble | Les $n$ nœuds du graphe (une ville = un nœud) |
| $\mathcal{E} \subseteq \mathcal{V} \times \mathcal{V}$ | Ensemble de paires | Les arêtes — chaque paire $(i,j)$ signifie "il existe un lien de $i$ vers $j$" |
| $\mathbf{X} \in \mathbb{R}^{n \times d_x}$ | Matrice réelle $n$ lignes $\times$ $d_x$ colonnes | La matrice de features des nœuds — la ligne $i$ contient les $d_x$ features de la ville $i$ |
| $\mathbf{E}_{ij} \in \mathbb{R}^{d_e}$ | Vecteur de taille $d_e$ | Les features de l'arête $(i,j)$ — ex. la distance euclidienne $\|x_i - x_j\|_2$ |
| $\mathbf{A} \in \{0,1\}^{n \times n}$ | Matrice binaire $n \times n$ | Matrice d'adjacence — $A_{ij} = 1$ si $(i,j)$ est une arête, $0$ sinon |

> **$\mathbb{R}^{n \times d}$** se lit "l'espace des matrices réelles à $n$ lignes et $d$ colonnes". Une matrice $\mathbf{X} \in \mathbb{R}^{n \times d}$ contient $n \times d$ nombres réels.

Pour le TSP, le graphe est **complet** ($|\mathcal{E}| = n(n-1)/2$) et **non orienté**.

**Pour le TSPTW**, le graphe est **orienté** (l'arc $i \to j$ est différent de $j \to i$ à cause des fenêtres temporelles), donc $|\mathcal{E}| = n(n-1)$.

#### Vecteur de features d'un nœud TSPTW

$$\mathbf{x}_i = (x_i,\; y_i,\; e_i,\; l_i,\; s_i)$$

| Symbole | Signification | Exemple |
|---------|---------------|---------|
| $x_i, y_i$ | Coordonnées géographiques de la ville $i$ | $(0.3, 0.7)$ |
| $e_i$ | *Earliest* — début de la fenêtre temporelle (heure d'ouverture) | $8.0$ (8h00) |
| $l_i$ | *Latest* — fin de la fenêtre temporelle (heure de fermeture) | $10.0$ (10h00) |
| $s_i$ | *Service time* — durée passée à servir le client une fois arrivé | $0.5$ (30 min) |

> En PyTorch : `x = torch.tensor([[x_i, y_i, e_i, l_i, s_i], ...])` de shape `[N, 5]`

Les **features d'arête** (distance, temps de trajet) sont stockées séparément dans `edge_attr` de shape `[E, D]`.

---

### Cadre MPNN (Message Passing Neural Network)

La mise à jour générale par passage de messages à la couche $l$ est :

$$\mathbf{m}_v^{(l)} = \bigoplus_{u \in \mathcal{N}(v)} \phi^{(l)}\!\left(\mathbf{h}_v^{(l-1)},\, \mathbf{h}_u^{(l-1)},\, \mathbf{e}_{uv}\right)$$

| Symbole | Signification |
|---------|---------------|
| $\mathbf{m}_v^{(l)}$ | Le **message agrégé** reçu par le nœud $v$ à la couche $l$ — c'est la somme de ce que tous ses voisins lui ont envoyé |
| $\bigoplus_{u \in \mathcal{N}(v)}$ | **Agrégation** sur tous les voisins $u$ de $v$ — peut être une somme $\sum$, une moyenne, ou un max |
| $\phi^{(l)}(\cdot)$ | La **fonction de message** — transforme les embeddings des deux nœuds $v$ et $u$ et de leur arête $e_{uv}$ en un vecteur-message |
| $\mathbf{h}_v^{(l-1)}$ | L'**embedding** du nœud $v$ à la couche précédente — vecteur qui résume tout ce que $v$ "sait" à l'instant $l-1$ |
| $\mathbf{e}_{uv}$ | La **feature d'arête** entre $u$ et $v$ (ex. distance entre les deux villes) |

$$\mathbf{h}_v^{(l)} = \psi^{(l)}\!\left(\mathbf{h}_v^{(l-1)},\, \mathbf{m}_v^{(l)}\right)$$

| Symbole | Signification |
|---------|---------------|
| $\mathbf{h}_v^{(l)}$ | Le **nouvel embedding** du nœud $v$ après la couche $l$ |
| $\psi^{(l)}(\cdot)$ | La **fonction de mise à jour** — combine l'ancien embedding et le message reçu pour produire le nouvel embedding |
| $\mathbf{h}_v^{(0)} = \mathbf{x}_v$ | L'**initialisation** — au départ, l'embedding d'un nœud est simplement ses features brutes |

> **Intuition :** après $k$ couches, l'embedding $\mathbf{h}_v^{(k)}$ encode non seulement les features de $v$, mais aussi le contexte de tout son voisinage à distance $k$.

---

### Variante GCN (Kipf & Welling, 2017)

$$\mathbf{H}^{(l+1)} = \sigma\!\left(\tilde{\mathbf{D}}^{-\frac{1}{2}}\,\tilde{\mathbf{A}}\,\tilde{\mathbf{D}}^{-\frac{1}{2}}\,\mathbf{H}^{(l)}\,\mathbf{W}^{(l)}\right)$$

| Symbole | Signification |
|---------|---------------|
| $\mathbf{H}^{(l)} \in \mathbb{R}^{n \times d}$ | Matrice de **tous les embeddings** à la couche $l$ — une ligne par nœud |
| $\tilde{\mathbf{A}} = \mathbf{A} + \mathbf{I}_n$ | Matrice d'adjacence avec **auto-boucles** ajoutées ($\mathbf{I}_n$ = matrice identité) — permet à chaque nœud de se "s'envoyer un message à lui-même" |
| $\tilde{\mathbf{D}}_{ii} = \sum_j \tilde{A}_{ij}$ | **Matrice de degré** — $\tilde{D}_{ii}$ est le nombre de voisins de $i$ (+ 1 pour l'auto-boucle) |
| $\tilde{\mathbf{D}}^{-\frac{1}{2}}$ | **Normalisation symétrique** — divise par $\sqrt{\text{degré}}$ pour éviter que les nœuds très connectés dominent |
| $\mathbf{W}^{(l)} \in \mathbb{R}^{d^{(l)} \times d^{(l+1)}}$ | **Matrice de poids apprise** à la couche $l$ — transforme les embeddings de dimension $d^{(l)}$ en dimension $d^{(l+1)}$ |
| $\sigma$ | **Fonction d'activation** non linéaire (ex. ReLU) — sans elle, empiler des couches équivaudrait à une seule transformation linéaire |

> **Limite du GCN :** tous les voisins contribuent **également** à la mise à jour. Le modèle ne peut pas ignorer un voisin peu pertinent.

---

### GAT — Graph Attention Network (Veličković, 2018)

Le GAT remplace la moyenne uniforme par des **poids d'attention appris** $\alpha_{ij}$ :

$$\mathbf{h}_i' = \sigma\left( \sum_{j \in \mathcal{N}(i)} \alpha_{ij} \cdot \mathbf{W} \cdot \mathbf{h}_j \right)$$

| Symbole | Signification |
|---------|---------------|
| $\alpha_{ij} \in [0,1]$ | **Poids d'attention** du voisin $j$ pour le nœud $i$ — appris automatiquement. $\sum_j \alpha_{ij} = 1$ |
| $\mathbf{W} \cdot \mathbf{h}_j$ | **Projection** de l'embedding de $j$ dans un nouvel espace (même $\mathbf{W}$ partagé pour tous les nœuds) |
| $\sum_{j} \alpha_{ij} \cdot \mathbf{W} \cdot \mathbf{h}_j$ | **Somme pondérée** des projections des voisins — les voisins importants ($\alpha$ élevé) contribuent davantage |

**Calcul de $\alpha_{ij}$ en 3 étapes :**

**Étape 1 — Score brut :**
$$e_{ij} = \text{LeakyReLU}\!\left( \mathbf{a}^\top \cdot [\mathbf{W}\mathbf{h}_i \,\|\, \mathbf{W}\mathbf{h}_j] \right)$$

| Symbole | Signification |
|---------|---------------|
| $[\mathbf{W}\mathbf{h}_i \,\|\, \mathbf{W}\mathbf{h}_j]$ | **Concaténation** des projections de $i$ et $j$ — vecteur de taille $2F'$ |
| $\mathbf{a} \in \mathbb{R}^{2F'}$ | **Vecteur de paramètres appris** — partagé pour toutes les paires |
| $\mathbf{a}^\top \cdot [\ldots]$ | **Produit scalaire** — réduit le vecteur $2F'$ en un seul scalaire $e_{ij}$ |
| $\text{LeakyReLU}$ | Activation qui laisse passer les valeurs négatives avec une pente faible (0.2) — évite les gradients nuls |

**Étape 2 — Normalisation :**
$$\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k \in \mathcal{N}(i)} \exp(e_{ik})}$$

| Symbole | Signification |
|---------|---------------|
| $\exp(e_{ij})$ | Exponentielle du score brut — transforme en valeur strictement positive |
| $\sum_{k \in \mathcal{N}(i)} \exp(e_{ik})$ | Somme des exponentielles sur **tous les voisins** de $i$ — sert de normalisation |
| $\alpha_{ij}$ | Score normalisé entre 0 et 1. Garantit $\sum_j \alpha_{ij} = 1$ (propriété du softmax) |

> **Softmax :** la formule $\frac{\exp(e_{ij})}{\sum_k \exp(e_{ik})}$ est appelée **softmax**. Elle convertit un ensemble de scores bruts en probabilités (valeurs positives qui somment à 1). Un score élevé $e_{ij}$ donnera un $\alpha_{ij}$ proche de 1 ; un score faible donnera un $\alpha_{ij}$ proche de 0.

**Attention multi-têtes ($K$ têtes en parallèle) :**
$$\mathbf{h}_i' = \Big\|_{k=1}^{K}\; \sigma\!\left( \sum_{j} \alpha_{ij}^k \cdot \mathbf{W}^k \cdot \mathbf{h}_j \right)$$

| Symbole | Signification |
|---------|---------------|
| $K$ | Nombre de **têtes d'attention** en parallèle — chacune apprend des poids différents |
| $\alpha_{ij}^k$ | Poids d'attention de la **tête $k$** pour la paire $(i,j)$ |
| $\mathbf{W}^k$ | Matrice de projection propre à la **tête $k$** |
| $\big\|_{k=1}^{K}$ | **Concaténation** des $K$ sorties — la dimension finale est $K \times F'$ |

> **Intuition multi-têtes :** chaque tête peut apprendre à se concentrer sur un critère différent. Tête 1 → proximité géographique. Tête 2 → compatibilité de fenêtres temporelles. Tête 3 → direction générale de la tournée. Sans qu'on le programme explicitement.

---

### TSP-GNN : Objectif de prédiction d'arêtes (Joshi et al., 2019)

**Mise à jour des arêtes :**
$$\mathbf{e}_{ij}^{(l+1)} = \text{ReLU}\!\left(\text{LN}\!\left(\mathbf{W}_1^{(l)}\,\mathbf{h}_i^{(l)} + \mathbf{W}_2^{(l)}\,\mathbf{h}_j^{(l)} + \mathbf{W}_3^{(l)}\,\mathbf{e}_{ij}^{(l)}\right)\right)$$

| Symbole | Signification |
|---------|---------------|
| $\mathbf{W}_1^{(l)}\,\mathbf{h}_i^{(l)}$ | Contribution du nœud **source** $i$ à l'arête |
| $\mathbf{W}_2^{(l)}\,\mathbf{h}_j^{(l)}$ | Contribution du nœud **destination** $j$ à l'arête |
| $\mathbf{W}_3^{(l)}\,\mathbf{e}_{ij}^{(l)}$ | **Mémoire** de l'arête — sa représentation à la couche précédente |
| $\text{LN}$ | **Layer Normalization** — normalise le vecteur pour stabiliser l'entraînement (évite les gradients explosifs) |
| $\text{ReLU}(x) = \max(0, x)$ | **Activation** — met à zéro les valeurs négatives, introduit la non-linéarité |

**Porte d'attention (η) :**
$$\hat{\eta}_{ij}^{(l)} = \text{softmax}_j\!\left(\text{gate}(\mathbf{e}_{ij}^{(l)})\right)$$

| Symbole | Signification |
|---------|---------------|
| $\text{gate}(\mathbf{e}_{ij})$ | Une couche linéaire $\mathbb{R}^d \to \mathbb{R}$ qui réduit l'embedding d'arête en un scalaire |
| $\text{softmax}_j$ | Normalisation sur **tous les voisins $j$ de $i$** — garantit $\sum_j \hat{\eta}_{ij} = 1$ |
| $\hat{\eta}_{ij}$ | Poids d'attention de l'arête $(i,j)$ — analogue à $\alpha_{ij}$ dans le GAT |

**Mise à jour des nœuds (résiduelle) :**
$$\mathbf{h}_i^{(l)} = \underbrace{\mathbf{h}_i^{(l-1)}}_{\text{résidu}} + \text{ReLU}\!\left(\text{LN}\!\left(\mathbf{W}_4^{(l)}\mathbf{h}_i^{(l-1)} + \sum_{j} \hat{\eta}_{ij}^{(l)} \cdot \mathbf{W}_5^{(l)} \mathbf{e}_{ij}^{(l)}\right)\right)$$

| Symbole | Signification |
|---------|---------------|
| $\mathbf{h}_i^{(l-1)}$ (résidu) | **Connexion résiduelle** — on ajoute l'ancien embedding au nouveau. Stabilise le gradient sur de nombreuses couches |
| $\mathbf{W}_4^{(l)}\mathbf{h}_i^{(l-1)}$ | **Auto-contribution** du nœud $i$ (ce qu'il sait déjà) |
| $\sum_{j} \hat{\eta}_{ij}^{(l)} \cdot \mathbf{W}_5^{(l)} \mathbf{e}_{ij}^{(l)}$ | **Agrégation pondérée** des arêtes voisines — les arêtes les plus pertinentes ($\hat{\eta}$ élevé) contribuent davantage |

**Sortie finale :**
$$\hat{p}_{ij} = \sigma\!\left(\text{MLP}\!\left(\mathbf{e}_{ij}^{(L)}\right)\right) \in [0,1]$$

| Symbole | Signification |
|---------|---------------|
| $\mathbf{e}_{ij}^{(L)}$ | Embedding de l'arête $(i,j)$ après **$L$ couches** — encode toute l'information accumulée |
| $\text{MLP}$ | *Multi-Layer Perceptron* — réseau dense $\mathbb{R}^d \to \mathbb{R}^{d/2} \to \mathbb{R}^1$ |
| $\sigma(x) = \frac{1}{1+e^{-x}}$ | **Sigmoïde** — écrase n'importe quel réel dans $[0,1]$. Ici : probabilité que l'arête $(i,j)$ appartienne à la tournée optimale |
| $\hat{p}_{ij}$ | **Probabilité prédite** pour l'arête $(i,j)$ — sortie finale du modèle |

**Perte d'entraînement — Entropie Croisée Binaire (BCE) :**
$$\mathcal{L} = -\frac{1}{|\mathcal{E}|} \sum_{(i,j)} \left[y_{ij}\,\log \hat{p}_{ij} + (1 - y_{ij})\,\log(1 - \hat{p}_{ij})\right]$$

| Symbole | Signification |
|---------|---------------|
| $y_{ij} \in \{0,1\}$ | **Label terrain** — 1 si l'arête $(i,j)$ est dans la tournée optimale, 0 sinon |
| $\hat{p}_{ij}$ | **Probabilité prédite** par le modèle |
| $\log \hat{p}_{ij}$ | Logarithme de la probabilité — élevé (proche de 0) si $\hat{p}_{ij}$ est proche de 1, très négatif si $\hat{p}_{ij}$ est proche de 0 |
| $y_{ij}\,\log \hat{p}_{ij} + (1-y_{ij})\,\log(1-\hat{p}_{ij})$ | Quand $y_{ij}=1$ : pénalise si $\hat{p}_{ij}$ est faible. Quand $y_{ij}=0$ : pénalise si $\hat{p}_{ij}$ est élevé |
| $-\frac{1}{|\mathcal{E}|}$ | Moyenne sur toutes les arêtes, avec signe $-$ pour transformer un maximum en minimum |

> **BCE en résumé :** pénalise le modèle quand il prédit une probabilité élevée pour une arête absente de la tournée, ou une probabilité faible pour une arête présente. Une BCE proche de 0 signifie que $\hat{p}_{ij} \approx y_{ij}$ pour toutes les arêtes.

## 3. Architecture / Étapes de l'algorithme

### Pipeline global

```
Graphe d'entrée G          Probabilités d'arêtes         Tournée TSP
 (coords nœuds)  →  Encodeur GNN  →  p̂_ij ∈ [0,1]  →  Décodeur  →  π
```

---

### Étape 1 — Encodage des entrées

Chaque nœud $i$ est projeté dans un espace de dimension $d$ :
$$\mathbf{h}_i^{(0)} = \mathbf{W}_{\text{in}}\,\mathbf{x}_i + \mathbf{b}_{\text{in}}$$

Chaque arête $(i,j)$ est initialisée avec sa distance euclidienne :
$$\mathbf{e}_{ij}^{(0)} = \mathbf{W}_{\text{arête}}\,\left[\,\|x_i - x_j\|_2\,\right] + \mathbf{b}_{\text{arête}}$$

---

### Étape 2 — $L$ couches de convolution sur graphe

Pour $l = 1, \dots, L$, répéter :

**2a. Mise à jour des arêtes** — combine les embeddings des nœuds incidents et l'état précédent de l'arête :
$$\mathbf{e}_{ij}^{(l)} = \text{ReLU}\!\left(\text{LN}\!\left(\mathbf{W}_1^{(l)}\mathbf{h}_i^{(l-1)} + \mathbf{W}_2^{(l)}\mathbf{h}_j^{(l-1)} + \mathbf{W}_3^{(l)}\mathbf{e}_{ij}^{(l-1)}\right)\right)$$

**2b. Porte d'attention** — sélection douce sur les arêtes voisines :
$$\hat{\eta}_{ij}^{(l)} = \frac{\exp\!\left(\text{gate}(\mathbf{e}_{ij}^{(l)})\right)}{\sum_{k \in \mathcal{N}(i)} \exp\!\left(\text{gate}(\mathbf{e}_{ik}^{(l)})\right)}$$

**2c. Mise à jour des nœuds** — agrégation résiduelle :
$$\mathbf{h}_i^{(l)} = \mathbf{h}_i^{(l-1)} + \text{ReLU}\!\left(\text{LN}\!\left(\mathbf{W}_4^{(l)}\mathbf{h}_i^{(l-1)} + \sum_{j \in \mathcal{N}(i)} \hat{\eta}_{ij}^{(l)} \cdot \mathbf{W}_5^{(l)} \mathbf{e}_{ij}^{(l)}\right)\right)$$

La connexion résiduelle $+ \mathbf{h}_i^{(l-1)}$ stabilise l'entraînement et facilite le flux de gradient sur de nombreuses couches.

---

### Étape 3 — Tête de scoring des arêtes

Après $L$ couches, un classificateur linéaire produit les logits d'arête :
$$\hat{p}_{ij} = \text{sigmoid}\!\left(\text{MLP}\!\left(\mathbf{e}_{ij}^{(L)}\right)\right) \in [0,1]$$

---

### Étape 4 — Décodage de la tournée

Le décodeur construit la tournée **ville par ville** de manière auto-régressive :

1. Regarder toutes les villes **non encore visitées**
2. Calculer un **score** pour chacune — comparaison entre un vecteur contexte et chaque embedding $h_i$
3. Sélectionner la ville selon la stratégie choisie :

| Décodeur | Description |
|----------|-------------|
| **Greedy** | Prend le score le plus élevé parmi toutes les villes non visitées |
| **Beam search** | Maintient $k$ tournées partielles en parallèle ; sélectionne les top-$k$ |
| **MCTS** | Monte Carlo Tree Search guidé par $\hat{p}_{ij}$ comme probabilités a priori |

4. Ajouter la ville à la route, la marquer visitée, mettre à jour le contexte
5. Recommencer jusqu'à toutes les villes visitées

---

### Procédure d'entraînement

**Supervisé** (faisable pour $n \leq 10$, labels calculés par force brute) :
```
Pour chaque époque :
  Pour chaque instance (G, π*) du jeu d'entraînement :
    1. Passe avant : calculer ĥ, ê à travers L couches GNN
    2. Calculer les probabilités d'arêtes p̂_ij
    3. Calculer la perte L = BCE(p̂, y*)   où y*_ij = 1 si (i,j) ∈ π*
    4. Rétropropagation à travers toutes les couches
    5. Mise à jour des poids avec Adam
```

**Reinforcement Learning (REINFORCE)** — utilisé quand les labels optimaux $\pi^*$ sont indisponibles (cas général pour $n > 10$) :

$$\nabla_\theta \mathcal{J}(\theta) = \mathbb{E}_{\pi \sim p_\theta}\!\left[(L(\pi) - b)\,\nabla_\theta \log p_\theta(\pi)\right]$$

où $L(\pi)$ est la longueur de la tournée (signal de récompense) et $b$ est une **ligne de base** (ex. rollout greedy) pour réduire la variance.

> **Pourquoi le RL plutôt que le supervisé ?** Calculer les solutions optimales pour le TSP est NP-difficile. Le solveur exact (Concorde) peut prendre des heures pour quelques centaines de villes. Avec le RL, on n'a jamais besoin de la solution optimale — le modèle apprend en comparant la qualité de ses propres tournées.

## 4. Analyse de Complexité

### Notation

| Symbole | Signification |
|---------|---------------|
| $n$ | nombre de nœuds (villes) |
| $|\mathcal{E}|$ | nombre d'arêtes |
| $L$ | nombre de couches GNN |
| $d$ | dimension d'embedding |
| $k$ | largeur du beam (décodeur) |

Pour un **graphe complet** (TSP), $|\mathcal{E}| = \dfrac{n(n-1)}{2} = O(n^2)$.  
Pour un **graphe orienté complet** (TSPTW), $|\mathcal{E}| = n(n-1) = O(n^2)$.

---

### Complexité temporelle

**Par couche GNN :**
$$T_{\text{arête}}^{(l)} = O(|\mathcal{E}| \cdot d^2) = O(n^2 d^2)$$
$$T_{\text{nœud}}^{(l)} = O(n \cdot (n-1) \cdot d) = O(n^2 d)$$

**Encodeur complet ($L$ couches) :**
$$T_{\text{encodeur}} = O(L \cdot n^2 \cdot d^2)$$

**Décodeur greedy :** $O(n^2)$

$$\boxed{T_{\text{total}} = O(L \cdot n^2 \cdot d^2)}$$

---

### Complexité spatiale

| Composant | Mémoire |
|-----------|---------|
| Embeddings de nœuds $\mathbf{H}^{(l)}$ | $O(L \cdot n \cdot d)$ |
| Embeddings d'arêtes $\mathbf{E}^{(l)}$ | $O(L \cdot n^2 \cdot d)$ |
| Matrices de poids (toutes couches) | $O(L \cdot d^2)$ |

Terme dominant : **embeddings d'arêtes** $O(L \cdot n^2 \cdot d)$, qui devient prohibitif pour de grands $n$.

$$\boxed{S_{\text{total}} = O(L \cdot n^2 \cdot d)}$$

---

### Comparaison avec d'autres méthodes

| Méthode | Temps (inférence) | Passe à $n = 10^4$ ? |
|---------|------------------|---------------------|
| Exact (Concorde) | $O(2^n)$ pire cas | Non |
| LKH-3 (méta-heuristique) | $O(n^2 \log n)$ par exécution | Difficilement |
| Transformer (Kool 2019) | $O(n^2 d)$ | Partiellement |
| **GNN (Joshi 2019)** | $O(L n^2 d^2)$ | Non (limité par la mémoire) |
| GNN creux | $O(L \cdot k_{\text{sp}} \cdot n \cdot d^2)$ | Oui (avec graphe $k$-NN creux) |

> **Note :** le goulot d'étranglement $O(n^2)$ peut être réduit en travaillant sur un graphe creux $k$-plus-proches-voisins, ramenant la complexité à $O(L \cdot k \cdot n \cdot d^2)$.

## 5. Forces et Limites

### Forces

| Propriété | Détail |
|-----------|--------|
| **Natif au graphe** | Encode directement la structure combinatoire (nœuds = villes, arêtes = connexions) sans aplatissement |
| **Invariance par permutation** | La sortie est invariante au relabeling des nœuds |
| **Supervision au niveau des arêtes** | Prédit *quelles arêtes* appartiennent à la tournée — signal fin qui généralise bien |
| **Inférence rapide** | Une passe avant produit les scores d'arêtes en millisecondes pour $n \leq 100$ |
| **Composable** | L'encodeur GNN peut être combiné avec n'importe quel décodeur (greedy, beam search, MCTS) |
| **Inductif** | Peut généraliser à des tailles de graphe non vues à l'entraînement (avec dégradation) |

---

### Limites

| Limite | Impact |
|--------|--------|
| **Mémoire quadratique** en $n$ | Les embeddings d'arêtes requièrent $O(n^2 d)$ de mémoire ; infaisable pour $n \gtrsim 1000$ sans approximation creuse |
| **Nécessite des données d'entraînement** | La variante supervisée a besoin de tournées optimales (ou proches) — coûteux à générer à grande échelle |
| **Écart à l'optimalité** | Typiquement 1–5 % au-dessus de l'optimal pour $n \leq 100$ ; l'écart croît avec $n$ |
| **Dégradation de la généralisation** | Les modèles entraînés sur $n=50$ performent mal sur $n=500$ sans réentraînement |
| **Sensibilité au décodeur** | La qualité finale dépend fortement de la stratégie de décodage |
| **Pas de respect strict des contraintes** | Le modèle peut produire des tournées invalides ; post-traitement nécessaire |

## 6. Cas d'Usage

### Quand utiliser un GNN pour l'optimisation combinatoire ?

Les GNN sont bien adaptés quand :

1. **Le problème est naturellement structuré en graphe** — villes, dépôts et routes forment un graphe. Les GNN exploitent cette topologie directement.

2. **Une résolution répétée sur des instances similaires est nécessaire** — le coût d'entraînement est amorti sur des milliers d'inférences.

3. **Des décisions quasi-temps-réel sont requises** — les solveurs classiques (Concorde, LKH) prennent des secondes à des minutes ; un GNN entraîné tourne en millisecondes.

4. **$n \leq 200$** — les GNN atteignent une qualité compétitive dans ce régime.

---

### Pourquoi les GNN spécifiquement (vs. Transformer ou Pointer Nets) ?

| Modèle | Biais inductif | Qualité TSP ($n=100$) |
|--------|---------------|----------------------|
| Pointer Network | Séquentiel, attention sur les nœuds | ~5 % d'écart |
| Transformer (Kool) | Attention par paires globale | ~1–2 % d'écart |
| **GNN (Joshi)** | Message passing local + features d'arêtes | ~1–3 % d'écart |

Les GNN ont un **prior structurel plus fort** : ils modélisent explicitement les arêtes (pas seulement les nœuds), ce qui s'aligne avec l'objectif TSP — sélectionner un sous-ensemble d'arêtes formant un cycle hamiltonien.

---

### Position dans le pipeline d'optimisation

```
Instance du problème
       │
       ▼
  Encodeur GNN          ← apprend les vraisemblances d'arêtes depuis la structure du graphe
       │
       ▼
  Scores d'arêtes p̂_ij
       │
       ├──► Décodeur greedy      (rapide, ~2–5 % d'écart)
       ├──► Beam search          (meilleure qualité, O(k·n²))
       └──► Recherche locale (2-opt, LKH)  ← utiliser la sortie GNN comme démarrage à chaud
```

## 7. Implémentation

Le code ci-dessous implémente le **TSPGNN** de Joshi et al. (2019) en PyTorch pur (sans PyTorch Geometric).

### Structure du code

| Fonction / Classe | Rôle |
|-------------------|------|
| `random_instance(n)` | Génère $n$ villes aléatoires dans $[0,1]^2$ |
| `tour_length(coords, tour)` | Calcule la longueur euclidienne totale d'une tournée fermée |
| `greedy_decode(p)` | Construit une tournée par greedy à partir des probabilités d'arêtes |
| `optimal_tour_labels(coords)` | Labels optimaux par force brute (faisable pour $n \leq 10$) |
| `GNNLayer` | Une couche résiduelle GCN avec mises à jour conjointes nœuds/arêtes |
| `TSPGNN` | Modèle complet : encodeur GNN + tête de classification d'arêtes |
| `train(model, n_nodes, n_steps)` | Entraînement supervisé par descente de gradient (Adam + BCE) |

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time
from itertools import permutations

# ── Data helpers ──────────────────────────────────────────────────────────────

def random_instance(n: int, seed: int = None) -> torch.Tensor:
    """n random cities uniformly distributed in [0, 1]²."""
    if seed is not None:
        torch.manual_seed(seed)
    return torch.rand(n, 2)


def tour_length(coords: torch.Tensor, tour: list) -> float:
    """Total Euclidean length of a closed tour."""
    idx = torch.tensor(tour + [tour[0]])
    return (coords[idx[:-1]] - coords[idx[1:]]).norm(dim=1).sum().item()


def greedy_decode(p: torch.Tensor, start: int = 0) -> list:
    """
    Greedy tour construction from edge probabilities.
    At each step, move to the highest-scoring unvisited city.
    """
    n = p.shape[0]
    visited = torch.zeros(n, dtype=torch.bool, device=p.device)
    tour = [start]
    visited[start] = True
    for _ in range(n - 1):
        scores = p[tour[-1]].clone()
        scores[visited] = -1.0
        next_city = scores.argmax().item()
        tour.append(next_city)
        visited[next_city] = True
    return tour


def optimal_tour_labels(coords: torch.Tensor) -> torch.Tensor:
    """
    Brute-force optimal tour for small instances (n ≤ 10).
    Returns a binary (n, n) edge matrix: y[i,j] = 1 iff (i,j) is in the optimal tour.
    """
    n = coords.shape[0]
    best_len, best_tour = float("inf"), None
    for perm in permutations(range(1, n)):
        tour = [0] + list(perm)
        length = tour_length(coords, tour)
        if length < best_len:
            best_len, best_tour = length, tour
    y = torch.zeros(n, n)
    for k in range(n):
        a, b = best_tour[k], best_tour[(k + 1) % n]
        y[a, b] = y[b, a] = 1.0
    return y


# ── Model ─────────────────────────────────────────────────────────────────────

class GNNLayer(nn.Module):
    """
    One residual GCN layer with joint edge and node updates (Joshi et al., 2019).

    Edge update:
        e_ij^(l) = ReLU(LN(W1·h_i + W2·h_j + W3·e_ij^(l-1)))

    Attention gate:
        η_ij^(l) = softmax_j( gate(e_ij^(l)) )

    Node update (residual):
        h_i^(l) = h_i^(l-1) + ReLU(LN(W4·h_i^(l-1) + Σ_j η_ij · W5·e_ij^(l)))
    """

    def __init__(self, d: int):
        super().__init__()
        # Edge weights
        self.W1 = nn.Linear(d, d, bias=False)   # h_i contribution
        self.W2 = nn.Linear(d, d, bias=False)   # h_j contribution
        self.W3 = nn.Linear(d, d, bias=False)   # e_ij contribution
        # Node weights
        self.W4 = nn.Linear(d, d, bias=False)   # self contribution
        self.W5 = nn.Linear(d, d, bias=False)   # aggregated edge contribution
        # Scalar attention gate: d → 1
        self.gate = nn.Linear(d, 1)
        # Layer norms (work on any batch/instance size, unlike BatchNorm)
        self.ln_e = nn.LayerNorm(d)
        self.ln_h = nn.LayerNorm(d)

    def forward(self, h: torch.Tensor, e: torch.Tensor):
        """
        h : (n, d)    node embeddings
        e : (n, n, d) edge embeddings
        """
        n, d = h.shape

        # ── Edge update ───────────────────────────────────────────────────────
        h_i  = self.W1(h).unsqueeze(1).expand(-1, n, -1)   # (n, n, d)
        h_j  = self.W2(h).unsqueeze(0).expand(n, -1, -1)   # (n, n, d)
        e_new = F.relu(self.ln_e(h_i + h_j + self.W3(e)))  # (n, n, d)

        # ── Attention gate ────────────────────────────────────────────────────
        eta = torch.softmax(self.gate(e_new), dim=1)        # (n, n, 1)

        # ── Node update (residual) ────────────────────────────────────────────
        agg   = (eta * self.W5(e_new)).sum(dim=1)           # (n, d)
        h_new = h + F.relu(self.ln_h(self.W4(h) + agg))    # (n, d)

        return h_new, e_new


class TSPGNN(nn.Module):
    """
    GNN encoder + edge classification head for TSP.

    Architecture:
      1. Linear projection: coords (n,2) → node embeddings (n,d)
      2. Linear projection: distances (n,n,1) → edge embeddings (n,n,d)
      3. L stacked GNNLayer blocks
      4. MLP edge head: (n,n,d) → (n,n) edge probabilities
    """

    def __init__(self, d: int = 128, L: int = 6):
        super().__init__()
        self.node_in  = nn.Linear(2, d)
        self.edge_in  = nn.Linear(1, d)
        self.layers   = nn.ModuleList([GNNLayer(d) for _ in range(L)])
        self.edge_out = nn.Sequential(
            nn.Linear(d, d // 2),
            nn.ReLU(),
            nn.Linear(d // 2, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : (n, 2)   node coordinates in [0, 1]²
        returns p : (n, n)  edge probabilities ∈ (0, 1)
        """
        h    = self.node_in(x)                                 # (n, d)
        dist = torch.cdist(x, x).unsqueeze(-1)                 # (n, n, 1)
        e    = self.edge_in(dist)                              # (n, n, d)

        for layer in self.layers:
            h, e = layer(h, e)

        p = torch.sigmoid(self.edge_out(e)).squeeze(-1)        # (n, n)
        return p


# ── Training ──────────────────────────────────────────────────────────────────

def train(model: TSPGNN, n_nodes: int = 8, n_steps: int = 500, lr: float = 1e-3):
    """
    Supervised training on randomly generated instances.
    Labels are computed by brute-force exact solver (feasible for n ≤ 10).

    Loss: binary cross-entropy over all n² edges.
    Optimizer: Adam.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []

    for step in range(n_steps):
        coords = random_instance(n_nodes)
        y      = optimal_tour_labels(coords)          # (n, n) ground truth
        p_hat  = model(coords)                        # (n, n) predictions

        loss = F.binary_cross_entropy(p_hat, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        if (step + 1) % 100 == 0:
            print(f"  step {step + 1:4d}/{n_steps}  BCE loss = {loss.item():.4f}")

    return losses


print("Model defined. Parameter counts:")
for d, L in [(64, 4), (128, 6)]:
    m = TSPGNN(d=d, L=L)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"  d={d:3d}, L={L}  →  {n_params:,} parameters")

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   - -------------------------------------- 4.5/114.6 MB 28.0 MB/s eta 0:00:04
   --- ------------------------------------ 11.3/114.6 MB 30.7 MB/s eta 0:00:04
   ------ --------------------------------- 17.8/114.6 MB 30.4 MB/s eta 0:00:04
   -------- ------------------------------- 24.6/114.6 MB 30.9 MB/s eta 0:00:03
   ---------- ----------------------------- 29.6/114.6 MB 29.6 MB/s eta 0:00:03
   ----------- ---------------------------- 33.3/114.6 MB 27.6 MB/s eta 0:00:03
   ------------- -------------------------- 39.1/114.6 MB 27.3 MB/s eta 0:00:03
   --------------- ------------------------ 45.6/114.6 MB 27.9 MB/s eta 0:00:03
   ----------------- ---------------------- 51.4/114.6 MB 28.2 MB/s eta 0:00:03
   -------------------- ------------------- 58.2/114.6 MB 28.4 MB/s eta 0:00:02
   ---------------------- ----------------- 64.2/114

## 8. Démonstration

> Parcours pas-à-pas d'une petite instance.

### Ce que fait la démonstration

La démo entraîne `TSPGNN(d=64, L=4)` sur des instances TSP à 8 villes avec **apprentissage supervisé** :

1. **Générer** une instance aléatoire : $n=8$ villes avec coordonnées dans $[0,1]^2$.
2. **Labelliser** : calculer la tournée optimale par force brute (faisable pour $n \leq 10$, seulement $(n-1)!/2 = 2520$ permutations à vérifier).
3. **Entraîner** : minimiser l'entropie croisée binaire entre les probabilités d'arêtes prédites $\hat{p}_{ij}$ et les labels terrain $y_{ij} \in \{0,1\}$.
4. **Évaluer** : sur une instance de test, décoder une tournée greedy et comparer sa longueur à l'optimal.

---

### Comment lire les 4 graphiques

| Graphique | Ce qu'il faut observer |
|-----------|------------------------|
| **Courbe de perte** | Doit diminuer régulièrement. Si elle se stabilise tôt, le modèle sous-apprend (augmenter `d` ou `L`). |
| **Labels terrain $y_{ij}$** | Matrice binaire creuse — seulement $n$ entrées non nulles (les arêtes de la tournée optimale). |
| **Probs prédites $\hat{p}_{ij}$** | Doit être élevé ($\approx 1$) sur les arêtes de la tournée, faible ($\approx 0$) ailleurs. |
| **Tournées** | Vert = optimal, bleu = greedy GNN. L'écart à l'optimalité devrait être < 5 % après 400 étapes sur $n=8$. |

---

### Pourquoi l'entraînement supervisé fonctionne ici (mais pas à grande échelle)

La labellisation par force brute est $O(n!)$, faisable uniquement pour $n \leq 10$. Pour de plus grandes instances, les labels optimaux ne peuvent pas être calculés efficacement (TSP est NP-difficile). Le **reinforcement learning** (REINFORCE) devient alors la seule stratégie d'entraînement viable :

$$\nabla_\theta \mathcal{J}(\theta) = \mathbb{E}_{\pi \sim p_\theta}\!\left[(L(\pi) - b)\,\nabla_\theta \log p_\theta(\pi)\right]$$

où $L(\pi)$ est la longueur de la tournée (signal de récompense) et $b$ est une ligne de base (ex. rollout greedy) pour réduire la variance. Aucune solution optimale n'est jamais nécessaire.

In [4]:
# ── Demo: supervised training on n=8, then walkthrough ───────────────────────

N_DEMO   = 8
D, L_GNN = 64, 4
N_STEPS  = 400

torch.manual_seed(42)
model_demo = TSPGNN(d=D, L=L_GNN)

print(f"Training TSPGNN(d={D}, L={L_GNN}) on n={N_DEMO} for {N_STEPS} steps …")
losses = train(model_demo, n_nodes=N_DEMO, n_steps=N_STEPS, lr=2e-3)
print("Done.\n")

# ── Inference on one held-out instance ────────────────────────────────────────
torch.manual_seed(7)
coords = random_instance(N_DEMO)
y_true = optimal_tour_labels(coords)

model_demo.eval()
with torch.no_grad():
    p_hat = model_demo(coords)

tour  = greedy_decode(p_hat)
length_gnn = tour_length(coords, tour)

# Compute optimal tour for comparison
best_len, best_tour = float("inf"), None
for perm in permutations(range(1, N_DEMO)):
    t = [0] + list(perm)
    l = tour_length(coords, t)
    if l < best_len:
        best_len, best_tour = l, t

gap = (length_gnn - best_len) / best_len * 100
print(f"Optimal tour length : {best_len:.4f}")
print(f"GNN greedy tour     : {length_gnn:.4f}  (optimality gap = {gap:.1f} %)")

# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# 1) Loss curve
axes[0].plot(losses, color="steelblue", lw=1.2)
axes[0].set_title("Training Loss (BCE)")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

# 2) Ground-truth edge labels
im0 = axes[1].imshow(y_true.numpy(), cmap="Greens", vmin=0, vmax=1)
axes[1].set_title("Ground-truth edges $y_{ij}$")
axes[1].set_xlabel("City $j$")
axes[1].set_ylabel("City $i$")
plt.colorbar(im0, ax=axes[1], fraction=0.046)

# 3) Predicted edge probabilities
im1 = axes[2].imshow(p_hat.numpy(), cmap="RdYlGn", vmin=0, vmax=1)
axes[2].set_title("Predicted probs $\\hat{p}_{ij}$")
axes[2].set_xlabel("City $j$")
axes[2].set_ylabel("City $i$")
plt.colorbar(im1, ax=axes[2], fraction=0.046)

# 4) Decoded tour vs optimal
xy = coords.numpy()

def draw_tour(ax, xy, tour, color, lw, label):
    tc = tour + [tour[0]]
    for k in range(len(tc) - 1):
        a, b = tc[k], tc[k + 1]
        ax.plot([xy[a, 0], xy[b, 0]], [xy[a, 1], xy[b, 1]],
                color=color, lw=lw, alpha=0.8, label=label if k == 0 else None)

draw_tour(axes[3], xy, best_tour, "green",    2.5, f"Optimal ({best_len:.3f})")
draw_tour(axes[3], xy, tour,      "steelblue", 1.5, f"GNN greedy ({length_gnn:.3f})")
axes[3].scatter(xy[:, 0], xy[:, 1], s=80, zorder=5, color="black")
for i, (xi, yi) in enumerate(xy):
    axes[3].annotate(str(i), (xi + 0.02, yi + 0.02), fontsize=9)
axes[3].set_title(f"Tours  (gap = {gap:.1f} %)")
axes[3].set_xlim(-0.05, 1.10)
axes[3].set_ylim(-0.05, 1.10)
axes[3].set_aspect("equal")
axes[3].legend(fontsize=8)
axes[3].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print(f"\nGNN tour  : {' → '.join(map(str, tour + [tour[0]]))}")
print(f"Optimal   : {' → '.join(map(str, best_tour + [best_tour[0]]))}")

Training TSPGNN(d=64, L=4) on n=8 for 400 steps …


AttributeError: module 'torch' has no attribute '_utils'

## 9. Benchmark

> Temps d'inférence et empreinte mémoire vs. taille de l'instance — modèle non entraîné (poids aléatoires).

### Ce qui est mesuré

| Métrique | Description |
|----------|-------------|
| **Temps d'inférence (ms)** | Temps pour une passe avant + décodage greedy, moyenné sur 20 répétitions |
| **Mémoire arêtes (MB)** | Taille du tenseur d'embedding d'arêtes $\mathbf{E} \in \mathbb{R}^{n \times n \times d}$ avec $d=128$, en float32 |

Note : le modèle utilise des **poids aléatoires** — cela isole le coût computationnel de la qualité de solution.

---

### Comment lire les 3 graphiques

| Graphique | Ce qu'il faut observer |
|-----------|------------------------|
| **Temps d'inférence** | Doit croître quadratiquement avec $n$ — chaque ville supplémentaire ajoute $n$ nouvelles arêtes à traiter. |
| **Échelle log-log** | La courbe mesurée doit suivre la ligne de référence $O(n^2)$. Une pente plus raide indiquerait un terme $O(n^3)$ caché. |
| **Empreinte mémoire** | Croît comme $n^2 \cdot d \cdot 4$ octets. À $n=200$, $d=128$ : $200^2 \times 128 \times 4 = 20.5$ MB — gérable. À $n=1000$ : $\approx 512$ MB par instance — problématique. |

---

### Limites pratiques

| $n$ | Mémoire arêtes ($d=128$) | Faisable ? |
|-----|-------------------------|-----------|
| 50  | ~1.3 MB | Oui |
| 100 | ~5.1 MB | Oui |
| 200 | ~20.5 MB | Oui |
| 500 | ~128 MB | Marginal |
| 1000 | ~512 MB | Non (risque OOM) |

Au-delà de $n \approx 300$, un **graphe $k$-NN creux** (garder uniquement les $k=10$ voisins les plus proches par nœud) réduit la mémoire à $O(k \cdot n \cdot d)$ — linéaire en $n$ — au prix d'une légère perte de qualité.

In [ ]:
# ── Benchmark: inference time and memory vs. instance size ───────────────────
# Note: model is untrained (random weights). We benchmark inference speed only.

model_bench = TSPGNN(d=128, L=6).eval()

SIZES  = [10, 20, 50, 100, 200]
N_REPS = 20          # repetitions per size for stable timing
results = {}

print(f"{'n':>6}  {'mean (ms)':>10}  {'std (ms)':>9}  {'edge mem (MB)':>14}")
print("-" * 46)

with torch.no_grad():
    for n in SIZES:
        times = []
        for rep in range(N_REPS):
            x  = random_instance(n)
            t0 = time.perf_counter()
            p  = model_bench(x)
            _  = greedy_decode(p)
            times.append(time.perf_counter() - t0)

        mean_ms = np.mean(times) * 1e3
        std_ms  = np.std(times)  * 1e3
        # Edge tensor (n, n, d=128) in float32 → bytes
        edge_mb = n * n * 128 * 4 / 1e6

        results[n] = {"mean_ms": mean_ms, "std_ms": std_ms, "edge_mb": edge_mb}
        print(f"{n:>6}  {mean_ms:>10.2f}  {std_ms:>9.2f}  {edge_mb:>14.1f}")

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ns    = SIZES
means = [results[n]["mean_ms"] for n in ns]
stds  = [results[n]["std_ms"]  for n in ns]
mems  = [results[n]["edge_mb"] for n in ns]

# 1) Inference time (linear scale)
axes[0].errorbar(ns, means, yerr=stds, marker="o", capsize=4, color="steelblue")
axes[0].set_xlabel("Number of cities $n$")
axes[0].set_ylabel("Inference time (ms)")
axes[0].set_title("GNN Inference Time")
axes[0].grid(True, alpha=0.3)

# 2) Log-log scaling vs O(n²) reference
ns_arr  = np.array(ns, dtype=float)
ref_n2  = means[0] * (ns_arr / ns_arr[0]) ** 2

axes[1].loglog(ns_arr, means,  "o-", color="steelblue", label="Measured")
axes[1].loglog(ns_arr, ref_n2, "r--",                   label=r"$O(n^2)$ reference")
axes[1].set_xlabel("$n$  (log)")
axes[1].set_ylabel("Time (ms, log)")
axes[1].set_title("Log-log Scaling")
axes[1].legend()
axes[1].grid(True, which="both", alpha=0.3)

# 3) Edge-embedding memory footprint
axes[2].plot(ns, mems, "o-", color="darkorange")
axes[2].set_xlabel("Number of cities $n$")
axes[2].set_ylabel("Edge embedding memory (MB)")
axes[2].set_title("Memory Footprint (edge tensors, $d=128$)")
axes[2].grid(True, alpha=0.3)

# Annotate quadratic growth
for n, m in zip(ns, mems):
    axes[2].annotate(f"{m:.1f} MB", (n, m), textcoords="offset points",
                     xytext=(5, 5), fontsize=8)

plt.tight_layout()
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
print("\nSummary — TSPGNN(d=128, L=6)")
print(f"{'n':>6}  {'time (ms)':>12}  {'edge mem':>10}  {'feasible?':>10}")
print("-" * 45)
for n in SIZES:
    r = results[n]
    feasible = "yes" if r["edge_mb"] < 500 else "no (OOM risk)"
    print(f"{n:>6}  {r['mean_ms']:>10.2f}ms  {r['edge_mb']:>8.1f}MB  {feasible:>10}")

## 10. Analyse Expérimentale

### Scalabilité

Le benchmark confirme le **goulot d'étranglement $O(n^2)$** prédit par l'analyse de complexité. Le coût dominant est le tenseur d'embedding d'arêtes $\mathbf{E}^{(l)} \in \mathbb{R}^{n \times n \times d}$, qui doit être alloué et mis à jour à chaque couche.

Deux atténuations existent :
- **Graphe creux** : restreindre les arêtes aux $k$ voisins les plus proches. Réduit la mémoire à $O(k \cdot n \cdot d)$, mais peut couper des arêtes longue-portée importantes.
- **GNN hiérarchique** : regrouper les villes en clusters, exécuter le GNN au sein des clusters puis entre clusters. Réduit le $n$ effectif à chaque niveau.

---

### Analyse comportementale

Après entraînement, les poids d'attention $\hat{\eta}_{ij}^{(l)}$ (porte d'attention dans `GNNLayer`) peuvent être visualisés pour comprendre ce que le modèle a appris :

- **Couches précoces** ($l=1,2$) : forte attention sur les voisins géographiquement proches — le modèle apprend la proximité locale.
- **Couches tardives** ($l=3,4$) : l'attention devient plus sélective, se concentrant sur les arêtes globalement cohérentes avec une tournée de faible coût (ex. arêtes qui ne "croisent" pas d'autres arêtes à score élevé).

Cela reflète ce que fait un solveur humain : identifier d'abord les arêtes candidates localement, puis filtrer pour la cohérence globale.

---

### Comparaison avec d'autres modèles

| Modèle | Architecture | Entraînement | Écart TSP $n=100$ | Mémoire |
|--------|-------------|--------------|-------------------|---------|
| **Pointer Network** (Vinyals 2015) | Seq2Seq + attention | RL (REINFORCE) | ~5 % | $O(n \cdot d)$ |
| **Transformer AM** (Kool 2019) | Encodeur-décodeur, attention multi-têtes | RL (REINFORCE + baseline greedy) | ~1–2 % | $O(n^2 \cdot d)$ |
| **GNN** (Joshi 2019) | GCN résiduel, embeddings d'arêtes | Supervisé (BCE) | ~1–3 % | $O(L \cdot n^2 \cdot d)$ |
| **LKH-3** (Helsgott) | Méta-heuristique | Aucun (recherche exacte) | < 0.5 % | $O(n^2)$ |

Enseignement clé : le GNN est **compétitif en qualité** avec les modèles basés sur les Transformers, mais sa mémoire $O(n^2)$ est une contrainte dure qui limite l'utilisation pratique à $n \leq 200$–$300$.

---

### Limitations

1. **Pas de respect des contraintes** — le modèle produit des probabilités d'arêtes, pas une tournée valide. Le décodeur greedy peut revisiter des villes ou violer des fenêtres temporelles (TSPTW). Un post-traitement (heuristique de réparation) est nécessaire.
2. **L'entraînement supervisé nécessite des labels** — pour $n > 10$, les labels optimaux doivent venir d'un solveur externe (LKH), ce qui est lent et limite la taille du jeu de données.
3. **Généralisation inter-tailles** — un modèle entraîné sur $n=50$ se dégrade significativement sur $n=200$. L'entraînement taille-invariant (mélanger les tailles d'instances) adresse partiellement ce problème.

---

### Propositions d'amélioration

| Proposition | Gain attendu |
|-------------|-------------|
| Passer à l'**entraînement RL** (REINFORCE) | Supprime le besoin de labels optimaux ; scalable à $n > 100$ |
| Utiliser un **graphe $k$-NN creux** | Réduit la mémoire de $O(n^2 d)$ à $O(k n d)$ ; permet $n > 500$ |
| Ajouter les **features nœuds TSPTW** $(x, y, e_i, l_i, s_i)$ | Étend le modèle aux instances avec fenêtres temporelles |
| Ajouter un **masque de faisabilité** dans le décodeur | Empêche de visiter les villes hors de leur fenêtre temporelle |
| **Post-traitement 2-opt** | Améliore la qualité de la tournée de ~1–2 % à faible coût |

## 11. Références

| Auteurs | Année | Titre | Revue/Conférence |
|---------|-------|-------|------------------|
| Scarselli et al. | 2009 | The Graph Neural Network Model | IEEE Trans. Neural Netw. |
| Kipf & Welling | 2017 | Semi-Supervised Classification with Graph Convolutional Networks | ICLR |
| Veličković et al. | 2018 | Graph Attention Networks (GAT) | ICLR |
| Vinyals et al. | 2015 | Pointer Networks | NeurIPS |
| Bello et al. | 2016 | Neural Combinatorial Optimization with Reinforcement Learning | NeurIPS Workshop |
| Kool et al. | 2019 | Attention, Learn to Solve Routing Problems! (Attention Model) | ICLR |
| Joshi et al. | 2019 | An Efficient Graph Convolutional Network Technique for the TSP | INFORMS |
| Solomon | 1987 | Algorithms for the Vehicle Routing and Scheduling Problems with Time Window Constraints | Operations Research |